# SFT の評価と比較動画

1. SFT（追加学習なし）を、学習後のモデルと同じ評価用の初期状態（100エピソード）で評価する
2. 3つのモデル（SFT、成功のみ、時間依存）を同じ初期状態で動かし、映像を横に並べた MP4 を作る

前提: `01_train_grpo.ipynb` で2つの条件の学習を終えていること。必要なもの: Colab の L4 GPU。SFT の評価は約30分、比較動画は約40分。

In [ ]:
import os
REPO_URL = "https://github.com/yutakambayashi/time-reward-grpo.git"  # このリポジトリの URL
OUT_ROOT = "/content/drive/MyDrive/art_embodied_runs"           # 学習と評価の出力先（Google Drive）

S_RUN, T_RUN, STEP = "grpo-success", "grpo-time", 40  # 比べるチェックポイント
os.environ.update(REPO_URL=REPO_URL, OUT_ROOT=OUT_ROOT, ART_OUTPUT_ROOT=OUT_ROOT, S_RUN=S_RUN, T_RUN=T_RUN, STEP=str(STEP))

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive', timeout_ms=600000)
!mkdir -p "$OUT_ROOT"

In [ ]:
%%bash
# ART-Embodied を検証した commit に固定して入れ、このリポジトリのパッチを当てる
set -e
pip install -q "uv>=0.12"
cd /content
[ -d ART-Embodied ] || git clone -q https://github.com/wandb/ART-Embodied.git
[ -d repo ] || git clone -q "$REPO_URL" repo
cd ART-Embodied
git checkout -q a37617ebfd9ced9455b6d4527a800f632fd0e37b
uv sync --python 3.12 --extra smolvla-libero 2>&1 | tail -1
python3 /content/repo/patches/patch_time_reward.py .

## 1. SFT の評価

In [ ]:
%%bash
set -e
cd /content/ART-Embodied
export MUJOCO_GL=egl PYOPENGL_PLATFORM=egl PYTHONUNBUFFERED=1 MPLBACKEND=Agg
uv run python /content/repo/scripts/make_cfg_train.py . /content/sft.yaml sft-eval 1 100 0
uv run python -m examples.embodied.libero.train_openvla_oft --evaluate-only --config /content/sft.yaml --evaluation-step 0 > /content/sft.log 2>&1
python3 /content/repo/scripts/summarize.py $OUT_ROOT/sft-eval/evaluation/update_000000_episode_outcomes.json

## 2. 比較動画

In [ ]:
%%bash
# 各タスクから2つの初期状態を選ぶ（全モデル成功で所要ステップ数の差が最大のもの、成否が分かれたもの）
E=evaluation/update_$(printf %06d $STEP)_episode_outcomes.json
python3 /content/repo/scripts/select_scenarios.py /content/scen.json \
  SFT=$OUT_ROOT/sft-eval/evaluation/update_000000_episode_outcomes.json S=$OUT_ROOT/$S_RUN/$E T=$OUT_ROOT/$T_RUN/$E

In [ ]:
%%bash
set -e
apt-get -qq install -y fonts-dejavu-core > /dev/null 2>&1
cd /content/ART-Embodied
export MUJOCO_GL=egl PYOPENGL_PLATFORM=egl PYTHONUNBUFFERED=1 MPLBACKEND=Agg
for name in sft s t; do
  uv run python /content/repo/scripts/make_cfg_train.py . /content/cmp_$name.yaml cmp-$name 1 100 0 > /dev/null
  uv run python - "$name" <<'PY'
import json, os, sys, yaml
name = sys.argv[1]
p = f"/content/cmp_{name}.yaml"
c = yaml.safe_load(open(p))
sc = json.load(open("/content/scen.json"))["scenarios"]
c["evaluation"]["episodes"] = len(sc)
c["evaluation"]["fixed_scenarios"] = sc
c["observability"]["videos_per_evaluation"] = len(sc)
c["storage"]["output_dir"] = f"{os.environ['OUT_ROOT']}/compare-videos/{name}"
yaml.safe_dump(c, open(p, "w"), sort_keys=False, allow_unicode=True)
PY
done
E="uv run python -m examples.embodied.libero.train_openvla_oft --evaluate-only"
$E --config /content/cmp_sft.yaml --evaluation-step 0 > /content/cmp_sft.log 2>&1
$E --config /content/cmp_s.yaml --evaluation-step $STEP --policy-checkpoint $OUT_ROOT/$S_RUN/checkpoints/step-$(printf %06d $STEP)/policy > /content/cmp_s.log 2>&1
$E --config /content/cmp_t.yaml --evaluation-step $STEP --policy-checkpoint $OUT_ROOT/$T_RUN/checkpoints/step-$(printf %06d $STEP)/policy > /content/cmp_t.log 2>&1
C=$OUT_ROOT/compare-videos
python3 /content/repo/scripts/build_compare.py /content/scen.json /content/compare SFT=$C/sft Success-only=$C/s Time-reward=$C/t
cd /content && zip -qr $OUT_ROOT/compare_videos.zip compare && echo "wrote $OUT_ROOT/compare_videos.zip"